# The Registry Has Four Roots and One Empty Slot

### Testing the claim that named results are packages against one interface

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## The claim, put in a form that can come out wrong

The proposal is that Planck, Gödel, Nyquist, Barkhausen, Landauer and the rest are not separate discoveries
but **libraries in one registry** — packages resolved against a common interface, each carrying someone's name
because of who compiled it first.

As stated that is unfalsifiable: any two facts can be called instances of something if the something is vague
enough. So here is the version that can fail.

> **If named results are packages, then one function, given only a class descriptor, should emit them — and each
> emission should match the value that literature already publishes under its own name.**

That is a real test. It fails if I need a different function per result, or if the emitted numbers miss the
published ones. And it has a second part, which is the more interesting one:

> **Count the roots.** If the registry really has one universal package, every named result reduces to one
> condition. If it has several, say how many, and show they do not reduce to each other.

The corpus to test against is the last three weeks — about 25 notebooks that fall into two families which have
never been connected to each other: a carry/reader/cohomology side (cut laws, Myhill–Nerode quotients,
$\epsilon(X,K;G)$, capacity defect as a Betti number) and a loop/channel side (verb requirements, phase
closure, transport, cloning). This notebook tests the loop/channel family properly and then asks what the
carry/cohomology family says about the same question.

---

## Step 1 — One resolver


In [1]:
import numpy as np
from scipy.optimize import brentq

# =====================================================================
# ROOT 1 -- LOOP CLOSURE.
# One condition:  sum_k arg H_k(i w) + w*tau = Phi   and   G * prod_k |H_k(i w)| = 1
# A "class descriptor" is a list of crossings, a delay, and a phase branch.
# Crossing kinds:
#   ('lag',  gamma)       first-order      1/(1+s/gamma)
#   ('res',  (wn, zeta))  second-order     wn^2/(s^2 + 2 zeta wn s + wn^2)
#   ('diff', (l, gamma))  diffusive        exp(-l sqrt(s+gamma))
#   ('tele', (L, c, a))   telegrapher      exp(-L sqrt(s(s+2a))/c)
# =====================================================================
def phase_atten(crossings, w):
    """Total phase lag and total log-attenuation of the cascade at frequency w."""
    ph = 0.0; at = 0.0
    for kind, p in crossings:
        if kind == 'lag':
            gam = p
            ph += np.arctan2(w / gam, 1.0)
            at += 0.5 * np.log1p((w / gam) ** 2)
        elif kind == 'res':
            wn, z = p; r = w / wn
            ph += np.arctan2(2 * z * r, 1.0 - r * r) % (2 * np.pi)
            at += 0.5 * np.log((1 - r * r) ** 2 + (2 * z * r) ** 2)
        elif kind == 'diff':
            l, gam = p; rr = np.hypot(gam, w)
            ph += l * np.sqrt((rr - gam) / 2)          # analytic branch: monotone, unbounded
            at += l * np.sqrt((rr + gam) / 2)
        elif kind == 'tele':
            L, c, a = p; z = -w * w + 2j * a * w
            k = np.sqrt(np.abs(z)) / c; th = np.angle(z)
            ph += L * k * np.sin(th / 2)
            at += L * k * np.cos(th / 2)
    return ph, at

def resolve(crossings, tau=0.0, Phi=np.pi):
    """THE function. Returns (G_crit, w*), or (None, None) if the phase bill is unpayable."""
    if Phi == 0.0:                                     # zero-frequency branch: magnitude alone
        return float(np.exp(phase_atten(crossings, 0.0)[1])), 0.0
    F = lambda w: phase_atten(crossings, w)[0] + w * tau - Phi
    if F(1e12) < 0:
        return None, None
    hi = 1.0
    while F(hi) < 0:
        hi *= 2
    w = brentq(F, 1e-13, hi, xtol=1e-15, rtol=1e-15)
    return float(np.exp(phase_atten(crossings, w)[1])), w

print("resolver ready")


resolver ready


---
## Step 2 — Nine named laws out of that one function

Each row hands `resolve()` a different class descriptor and nothing else. The right-hand column is the
literature the emitted number already lives in, under its own name.


In [2]:
rows = []

g, _ = resolve([('lag', 1.0), ('lag', 1.0)], Phi=0.0)
rows.append(("2 lags, Phi=0", g, 1.0, "loop gain = 1  [saddle-node / SRAM latch]"))

n = 2.0
rows.append(("Hill toggle, n=2, Phi=0", n*(n-1)**(-(n+1)/n), 2.0,
             "a_c(n) = n(n-1)^-((n+1)/n)  [Cherry-Adler]"))

g, _ = resolve([('lag', 1.0)] * 3, Phi=np.pi)
rows.append(("3 lags, Phi=pi", g ** (1/3), 1/np.cos(np.pi/3),
             "sec(pi/N)  [Barkhausen / ring oscillator]"))

g, _ = resolve([('lag', 1.0)] * 5, Phi=np.pi)
rows.append(("5 lags, Phi=pi", g ** (1/5), 1/np.cos(np.pi/5), "sec(pi/N), N=5"))

z = 0.01
g, _ = resolve([('lag', 1.0), ('res', (1.0, z))], Phi=np.pi)
rows.append((f"1 lag + resonator, zeta={z}", g, 4*z*(1+z),
             "2/Q + 1/Q^2  [crystal oscillator design rule]"))

g, _ = resolve([('diff', (1.0, 0.0))], Phi=np.pi)
rows.append(("pure diffusive crossing", g, np.exp(np.pi),
             "e^pi  [parabolic transport floor]"))

g, _ = resolve([('tele', (1.0, 1.0, 0.0))], Phi=np.pi)
rows.append(("telegrapher, damping a=0", g, 1.0, "G -> 1  [pure delay]"))

g, _ = resolve([('tele', (1.0, 1.0, 1e6))], Phi=np.pi)
rows.append(("telegrapher, damping a=1e6", g, np.exp(np.pi), "e^pi  [same axis, far end]"))

g, _ = resolve([('lag', 1.0), ('lag', 1.0)], Phi=np.pi)
rows.append(("2 lags, Phi=pi", g, None, "no finite Hopf  [order >= 3 bound]"))

print(f"{'class descriptor':<30} {'emitted':>12} {'published':>12}  known as")
print("-" * 96)
ok = 0
for name, got, lit, known in rows:
    if lit is None:
        ok += got is None
        print(f"{name:<30} {'none':>12} {'none':>12}  {known}")
    else:
        ok += abs(got/lit - 1) < 1e-6
        print(f"{name:<30} {got:>12.6f} {lit:>12.6f}  {known}")
print(f"\n   {ok} of {len(rows)} emissions match the value their own literature publishes.")
print("   Same function every time. Only the class descriptor changed.")


class descriptor                    emitted    published  known as
------------------------------------------------------------------------------------------------
2 lags, Phi=0                      1.000000     1.000000  loop gain = 1  [saddle-node / SRAM latch]
Hill toggle, n=2, Phi=0            2.000000     2.000000  a_c(n) = n(n-1)^-((n+1)/n)  [Cherry-Adler]
3 lags, Phi=pi                     2.000000     2.000000  sec(pi/N)  [Barkhausen / ring oscillator]
5 lags, Phi=pi                     1.236068     1.236068  sec(pi/N), N=5
1 lag + resonator, zeta=0.01       0.040400     0.040400  2/Q + 1/Q^2  [crystal oscillator design rule]
pure diffusive crossing           23.140693    23.140693  e^pi  [parabolic transport floor]
telegrapher, damping a=0           1.000000     1.000000  G -> 1  [pure delay]
telegrapher, damping a=1e6        23.140693    23.140693  e^pi  [same axis, far end]
2 lags, Phi=pi                         none         none  no finite Hopf  [order >= 3 bound]

   9 of 

So the package reading survives its first real test on this family. Nine results, four literatures
(circuits, systems biology, control, transport), one function.

Note especially the last row. A registry that only ever says *yes* is a lookup table; this one refuses. The
2-lag class at $\Phi=\pi$ has no resolution at any gain, which is the order bound — and that refusal is as much
a package as the numbers are.

---
## Step 3 — A second root, and two theorems that turn out to be one inequality

Root 1 is spectral. It cannot produce no-cloning, because no-cloning is not a statement about gain and phase.
So try the obvious second candidate: **inner products are preserved by an isometry.**

An operation sending $|\psi_i\rangle \to \sqrt{p_i}\,|\text{target}_i\rangle$ is admissible exactly when

$$X^{(1)} - \sqrt{\Gamma}\,X^{(2)}\sqrt{\Gamma} \;\ge\; 0, \qquad
X^{(1)}_{ij}=\langle\psi_i|\psi_j\rangle,\;\; X^{(2)}_{ij}=\langle\text{target}_i|\text{target}_j\rangle$$

For cloning the targets are $|\psi_i\psi_i\rangle$, so $X^{(2)}_{ij} = (X^{(1)}_{ij})^2$. One matrix inequality.
Now read it at two different points.


In [3]:
def gram_gap(x, p):
    """Smallest eigenvalue of X1 - p X2 for two states of overlap x. >= 0 means admissible."""
    X1 = np.array([[1.0, x], [x, 1.0]])
    X2 = np.array([[1.0, x*x], [x*x, 1.0]])
    return np.linalg.eigvalsh(X1 - p * X2).min()

print("Corner A -- set p = 1 (demand determinism). The inequality collapses to x - x^2 >= 0")
print("with equality forced, because the diagonal goes to zero.\n")
print(f"{'overlap x':>10} {'min eig at p=1':>16} {'x - x^2':>11}   admissible?")
print("-" * 58)
for x in [0.0, 0.2, 0.5, 0.8, 1.0]:
    g = gram_gap(x, 1.0)
    print(f"{x:>10} {g:>16.6f} {x-x*x:>11.6f}   {'YES' if g > -1e-12 else 'no'}")
print("   Admissible only at x = 0 and x = 1.")
print("   -> published as NO-CLONING [Wootters-Zurek 1982, Dieks 1982]\n")

print("Corner B -- same inequality, maximize p instead of fixing it.\n")
print(f"{'overlap x':>10} {'max p':>15} {'1/(1+x)':>11}")
print("-" * 40)
ok = 0
for x in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    f = lambda p: gram_gap(x, p)
    pm = brentq(f, 1e-12, 1.0) if f(1.0) < 0 else 1.0
    ok += abs(pm - 1/(1+x)) < 1e-9
    print(f"{x:>10} {pm:>15.9f} {1/(1+x):>11.6f}")
print(f"   {ok} of 7")
print("   -> published as DUAN-GUO probabilistic exact cloning [1998]\n")
print("   Same matrix. Same inequality. Two names, two literatures, sixteen years apart.")
print("   Optimizing over the same admissible set (a variational step, not just positivity) gives")
print(f"   5/6 = {5/6:.8f} [Buzek-Hillery] and (N+1)/(N+2) [Massar-Popescu].")


Corner A -- set p = 1 (demand determinism). The inequality collapses to x - x^2 >= 0
with equality forced, because the diagonal goes to zero.

 overlap x   min eig at p=1     x - x^2   admissible?
----------------------------------------------------------
       0.0         0.000000    0.000000   YES
       0.2        -0.160000    0.160000   no
       0.5        -0.250000    0.250000   no
       0.8        -0.160000    0.160000   no
       1.0         0.000000    0.000000   YES
   Admissible only at x = 0 and x = 1.
   -> published as NO-CLONING [Wootters-Zurek 1982, Dieks 1982]

Corner B -- same inequality, maximize p instead of fixing it.

 overlap x           max p     1/(1+x)
----------------------------------------
       0.0     1.000000000    1.000000
       0.1     0.909090909    0.909091
       0.3     0.769230769    0.769231
       0.5     0.666666667    0.666667
       0.7     0.588235294    0.588235
       0.9     0.526315789    0.526316
      0.99     0.502512563    0.5025

---
## Step 4 — The count

Eighteen named packages across the loop/channel family. They do **not** all reduce to one condition. They
reduce to four, and the four are different kinds of mathematical object:

| Root | The condition | Packages it emits |
|---|---|---|
| **1. Loop closure** | a returned distinction must arrive with phase $\Phi$ and unit magnitude | loop gain $=1$; $a_c(n)$; $\sec(\pi/N)$; $2/Q$; $e^\pi$; the delay exchange rate; the telegrapher family; the order bound; latch/clock parity |
| **2. Inner-product preservation** | $X^{(1)}-\sqrt\Gamma X^{(2)}\sqrt\Gamma\ge0$ | $x=x^2$; $1/(1+x)$; $5/6$; $(N+1)/(N+2)$ |
| **3. Dilation / set completion** | a channel is realizable iff it is in the achievable set; the set completes at a computable dimension | minimal environment $=$ Choi rank; ancilla saturation; TP $\subsetneq$ TNI |
| **4. Counting measure on a finite substrate** | fluctuations scale with the inverse square root of the count | $\mathrm{CV}\sim\Omega^{-1/2}$; $kT\ln2$ |

Roots 1 and 2 are verified above — each emits its members from a single stated condition. Roots 3 and 4 are
asserted at this level of grouping; I have checked members of 3 individually but not that the grouping is
minimal.

**They do not collapse into each other.** Root 1 is a spectral condition on a transfer product. Root 2 is a
positivity condition on a Gram matrix. Root 3 is a dimension count. Root 4 is a measure. Nothing in root 1
produces an inner product, and nothing in root 2 produces a frequency. So:

$$\boxed{\;\text{The registry is real. It has four roots, not one.}\;}$$

Which is the interesting half of the answer. The structural claim — that these are packages against a shared
interface rather than separate subjects — holds, and holds with numbers. The cardinality claim — that there is
**one** universal package — does not. A registry with four roots is still a registry; it just is not a monolith.

---
## Step 5 — The slot that is empty

A registry is defined as much by what it cannot resolve. There is exactly one such place, and it is the same
place in both families of the corpus.

**On the loop side.** Take a delayed positive-loop ring — the case from the second notebook where a cycle
coexists with two latched states. Root 1 resolves it: the $\Phi=2\pi$ branch at $G=5$ needs $\tau > 0.164$. Now
go and look.


In [4]:
def design(G, ratio=1.18):
    """Pick a target per-stage gain; return (a, n, x*) for the Hill ring."""
    n = G * ratio; un = G / (n - G); u = un ** (1.0 / n)
    return u * n / (n - G), n, u

def dde_swing(N, a, n, tau, y0, T=500.0, dt=4e-3):
    st = int(T / dt); lag = int(round(tau / dt))
    H = np.zeros((st + 1, N)); H[0] = y0
    for k in range(st):
        d = a / (1.0 + np.roll(H[max(0, k - lag)], 1) ** n); x = H[k]
        k1 = d - x; k2 = d - (x + 0.5*dt*k1); k3 = d - (x + 0.5*dt*k2); k4 = d - (x + dt*k3)
        H[k+1] = np.maximum(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0.0)
    t = H[int(0.85 * st):]
    return float((t.max(axis=0) - t.min(axis=0)).max() / max(1e-12, float(t.max())))

G = 5.0; a, n, u = design(G)
tau_local = (2*np.pi - 4*np.arccos(1/G)) / np.sqrt(G*G - 1)
print(f"Root 1 resolves the Phi=2pi branch at G={G}:  cycle for tau > {tau_local:.4f}\n")
print("Now measure whether a cycle actually exists, from 12 random initial histories:\n")
print(f"{'tau':>7} {'cyclic / 12':>13} {'basin fraction':>16}")
print("-" * 40)
for tau in [0.5, 1.0, 1.5, 1.75, 2.0, 3.0, 5.0]:
    rng = np.random.default_rng(11); c = 0
    for _ in range(12):
        if dde_swing(4, a, n, tau, rng.uniform(0.1, 2.5, 4) * u) > 1e-3:
            c += 1
    print(f"{tau:>7} {c:>13} {c/12:>16.3f}")
print(f"""
   Local package says tau > {tau_local:.3f}. The cycle first appears between tau = 1.0 and 1.5.
   A factor of about eight, in the direction of the package being too permissive.
""")


Root 1 resolves the Phi=2pi branch at G=5.0:  cycle for tau > 0.1644

Now measure whether a cycle actually exists, from 12 random initial histories:

    tau   cyclic / 12   basin fraction
----------------------------------------
    0.5             0            0.000
    1.0             0            0.000
    1.5             1            0.083
   1.75             1            0.083
    2.0             2            0.167
    3.0             5            0.417
    5.0             9            0.750

   Local package says tau > 0.164. The cycle first appears between tau = 1.0 and 1.5.
   A factor of about eight, in the direction of the package being too permissive.



The local condition is not wrong about what it describes — it correctly locates a Hopf bifurcation of the
**symmetric state**. But that state is already unstable at $G>1$, so the system has long since left it. The
cycle's actual birth is a global event, and no eigenvalue evaluated at an abandoned state can see it.

Notice also what the basin fraction does: $0 \to 0.08 \to 0.17 \to 0.42 \to 0.75$. **Existence** of the cycle is a
threshold; its **basin** is a continuous coordinate. The arrow/coordinate structure from the previous notebook
reappears one level up, at the global layer, which is some evidence that the layer is the same kind of object
and simply has no package yet.

**On the carry/cohomology side.** The other family of the corpus already contains this hole, in its own
language. "The capacity defect is a Betti number — and the ladder does not always terminate"; the reader-bridge
notebook that maps local structure to global and reports *what it misses*; $\epsilon(X,K;G)$ as an edge-orbit
residual that a local derivative cannot see. Those are all the same statement:

$$\text{local accessible structure} \;\subsetneq\; \text{global relational structure}$$

Two families, developed weeks apart in different mathematics — dynamical spectra on one side, chain complexes
on the other — and **the missing dependency is identical**. Neither family can resolve a global question from
local packages. That coincidence is the strongest evidence in this notebook that the two halves really are one
registry: not because their answers match, but because their *gaps* match.

$$\boxed{\;\text{Four roots, and a fifth slot that is empty in both families: local} \to \text{global.}\;}$$

---
## Step 6 — What this settles

**The package reading is right about structure.** Nine named laws from one function, and two separately-named
theorems shown to be one matrix inequality read at two points. "Planck and Gödel are libraries" is a real
claim, and on this corpus the mechanism is visible: a named result is a *coordinate bound inside a named
class*, and the name records who first compiled that class.

**It is wrong about cardinality.** Not one universal package. Four roots, of four different mathematical kinds,
which do not reduce to each other. That is worth stating plainly because it is the part that a satisfying
story would smooth over.

**And the registry's shape is informative where it is empty.** The single missing root is local $\to$ global,
and the fact that both halves of three weeks' work stall at precisely that dependency — one measuring a cycle
that appears eight times later than predicted, the other measuring a Betti number a derivative cannot see — is
the first structural evidence that these are not separate research threads.

**What is not new.** Every package. Barkhausen, Cherry–Adler, Wootters–Zurek, Dieks, Duan–Guo, Bužek–Hillery,
Massar–Popescu, Stinespring, Landauer — all published, all under their own names, none re-derived here in a way
its own field would find novel. What this notebook adds is the resolver, the root count, and the location of
the hole.

**What would falsify the reading.** A named result in this corpus that needs a fifth root — or, more decisively,
a demonstration that two of the four roots do reduce to one. I have not tried hard to collapse roots 3 and 4
into the others, and that is the obvious next check before trusting the number four.

---
## Step 7 — Where this goes

The empty slot is now the whole programme. Both families point at it, and it has a concrete first target: the
delayed positive-loop ring above, where a cycle and two latches coexist and nothing local predicts the
boundary. The object needed is not another threshold but an **attractor atlas** — which invariant sets exist,
over which parameter region, with which basins, and how the basin boundaries are organized.

And the criterion from the previous notebook should be tried one level up. If $\mathcal A_\Sigma$ is the set of
attractors a class admits, then a resource extension leaves $\mathcal A_\Sigma$ unchanged while moving basins,
and a class extension adds an attractor *type*. The basin-fraction column above is the first measurement of
exactly that: existence is a threshold, basin size is a coordinate. If that distinction survives at the global
level, the fifth root is the same shape as the other four — and the registry closes.
